# Manual Pipeline 2
- In this notebook, I will refactor the manual_pipeline1 processing into a single function.
- I will have an option for autopower or n_points as part of the function.
- Add function that tests for closeness between selected peaks:
    - Consider the LSP resolution.
- Afterwards cluster peaks in the same ranges and choose the strongest like in the priors analysis.

In [2]:
import pandas as pd
from astropy.time import Time
import matplotlib.pyplot as plt
import celerite2
import numpy as np
from celerite2 import terms
from scipy.optimize import minimize
from prettytable import PrettyTable
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks
from scipy.optimize import curve_fit


# Importing data
data = pd.read_csv(r"./Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)

# data = pd.read_csv(r"./Data/hd166_caii.txt", sep='\s',skip_blank_lines=True)
data.columns = ["JD", "sind"]
data['JD'] = data['JD'] + 2400000.0 # this file drops the 24 prefix
time_obj = Time(data["JD"].to_numpy(), format='jd', scale='tdb')
data["year"] = time_obj.jyear
data["day"] = time_obj.jd 
data['datetime'] = time_obj.to_datetime(timezone=None)
data = data.set_index('datetime')

# No need to down sample the dataset here. yay!
def split_df(df, train_split=0.8, valid_split=0.19):
    n = len(df)
    train_idx = round(train_split * n)
    valid_idx = round((train_split + valid_split) * n)
    return df.iloc[:train_idx].copy(), df.iloc[train_idx:valid_idx].copy(), df.iloc[valid_idx:].copy()

# Split the data
train_df, valid_df, test_df = split_df(data)


C:\Users\Joey\AppData\Local\Temp\ipykernel_27420\3617134450.py:15: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  data = pd.read_csv(r"./Data/benchmark/HD81809_Mt_wilson_data.txt", sep='\s',skip_blank_lines=True)


In [6]:
def check_aliases(freq, accepted_freqs, tol = 0.05):
    '''
    Check if a single period is an alias.
    If it is an alias we can skip it and use the next period as dominant.
    Aliases are not subtracted: they are noise.
    If True, signal is not an alias so set it be added.
    '''
    period = 1/freq
    accepted_periods = 1/accepted_freqs
    for p in accepted_periods:
        for mult in [0.25, 0.33, 0.5, 2, 3, 4]:
            delta = np.abs((period - p*mult)/period)
            if delta < tol: # The signal is an alias
                return False, p, mult, delta
    return True, None, None, None

def check_windows(freq, tol = 0.05):
    '''
    Checks if a signal looks suspiciously like a window function.
    If True, signal is not a window as defined.
    '''
    period = 1/freq
    year = 365.25
    month = year /12
    week = 7
    windows = [year, year/2, year/3, year/4, month, 2*month, week, 2* week, 3 * week]
    for window in windows:
        delta = np.abs((period-window)/period)
        if delta < tol: # Then we classify the signal as a window function
            return False, window, delta
    return True

In [ ]:
# Helpers for performing the SNR checks
def gaussian(t, A, mu, std, c):
    return A * np.exp(-0.5 * ((t - mu) / std)**2) + c

def find_snr(powers, freqs, peak_freq, peak_idx, peak_height, window = 150):
    
    # Define a window around the peak to which to fit a Gaussian
    n = len(powers)
    left  = max(0, peak_idx - window)
    right = min(n, peak_idx + window + 1)
    window_freqs = freqs[left:right]
    window_powers = powers[left:right]
    minf = min(window_freqs)
    maxf = max(window_freqs)

    # Define initial guesses for the fit
    A0 = peak_height
    mu0 = peak_freq
    std0 = mu0 * 0.1 #FWHM is a bit overkill
    c0 = np.median(window_powers)
    p0 = [A0, mu0, std0, c0]

    # Try the curve fit
    try:
        popt, _ = curve_fit(gaussian, window_freqs, window_powers,
                        p0 = p0,
                        bounds = ([0, minf, 0, 0], [np.inf, maxf, np.inf, np.inf]))
        resid_lsp = window_powers - gaussian(window_freqs, *popt)
        noise_std = np.std(resid_lsp)
        snr = popt[0] / noise_std # popt[0] is the A[0]
        return snr, popt, resid_lsp
    except (RuntimeError, ValueError):
        noise = np.mean(window_powers)
        return peak_height/noise, None, None


In [ ]:
def find_peaks(df,
            manual_freq = None, period_range = [0.1, 100*365], n_periods = 10000, 
            FAPs = [10,5,1,0.1], key_FAP_idx = -1, 
            threshold = 5,
            plot_LSP = False, plot_resids = False):
    '''
    Iteratively finds the peaks of the LSP.
    This will be used in the prior selsection process.
    Finds one peak then checks if it is an alias or a window.
    Also checks if it is too similar to a previous peak for the LSP to have resolved.
    If that peak also satiesfies an SNR threshold, add it to an accepted periods list.

    df is the dataframe being processed (training set)
    manual_freq True means the period_range and n_periods defines the set of periods calculated.
    Period_range is in days

    Returns a list of accepted periods.
    '''
    accepted_peak_freqs = [] # We will work in frequency space for the analysis other than for the plotting. It is easier for the SNR fitting.

    if manual_freq == 'linear':
        min_period = period_range[0]
        max_period = period_range[1]
        periods = np.linspace(min_period, max_period, n_periods)
        freqs = 1 / periods
    if manual_freq == 'log':
        min_period = period_range[0]
        max_period = period_range[1]
        periods = np.logspace(np.log(min_period), np.log(max_period), n_periods)
        freqs = 1 / periods
    
    resids = [df['sind']]
    i = 0 # What index in resids we are presently processing
    t = df['day'] # Fixed time axis for al

    while True:
        # Take the LSP : powers for the freqs
        ls = LombScargle(t, resids[i])
        if manual_freq is not None: # Use autopower
            freqs, powers = ls.autopower()
        else:
            powers = ls.power(freqs)

        # Now calculate the FAPs
        FAPs = np.array(FAPs)/100
        power_invFAPs = ls.false_alarm_level(FAPs, method = 'bootstrap') 
        key_FAP = float(FAPs[key_FAP_idx])

        # Identify FAPs above the key FAP
        peak_idxs, properties = find_peaks(powers, height=key_FAP) # peaks is an array of the indices of the input arrays with the peaks
        peak_freqs = freqs[peak_idxs]
        peak_heights = properties['peak_heights']

        # Break if none
        if len(peak_freqs) == 0:
            break

        # Sort identified peaks by height
        sort_idx = np.argsort(peak_heights)
        peak_freqs = peak_freqs[sort_idx]
        peak_heights = peak_heights[sort_idx] 

        # Loop through peaks until one satisfies conditions or the end is reached
        for peak_freq, peak_height, peak_idx in zip(peak_freqs, peak_heights, peak_idxs):
            alias_cond, p, mult, delta = check_aliases(freq = peak_freq, accepted_freqs= accepted_peak_freqs)
            if alias_cond == False: # This is an alias; skip freq
                print(f"Peak at freq {peak_freq} was identified to be an alias of {p} days at mult {mult} by tolerance {delta}.")

            else: # Not an alias; now check window
                window_cond, window, delta = check_windows(freq = peak_freq)
                if window_cond == False: # This is a window function; skip freq
                    print(f"Peak at freq {peak_freq} was identified to be a window of {window} days by tolerance {delta}.")
                
                else: # Not a window; now check SNR
                    snr, popt, resid_lsr =  find_snr(powers, freqs, peak_freq, peak_idx, peak_height, window = 150)
                    if resid_lsr is None:
                        print("SNR failed to fit Gaussian. Fell back to height vs average.")

                    if snr > threshold: # All checks passed. Append
                        



            
    
    # Now we have a list of freqs: plot and return
    
        
        
    
    
    

